# Arrhythmia Types - Real ECG Examples

One real patient recording per arrhythmia type, from the datasets used in
the thesis (PTB-XL, Chapman-Shaoxing, MIT-BIH, afdb, vfdb).

All signals are real clinical recordings, not simulated.


In [ ]:
# packages
%pip install -q wfdb neurokit2 matplotlib

In [ ]:
# helpers: download a record and plot 10 seconds of lead II
import urllib.request, os, wfdb
import numpy as np, matplotlib.pyplot as plt

PTBXL = 'https://physionet.org/files/ptb-xl/1.0.3/'
CHAP  = 'https://physionet.org/files/ecg-arrhythmia/1.0.0/'

def ptbxl(ecg_id):
    folder = (ecg_id - 1) // 1000 * 1000
    path = f'ptbxl/records100/{folder:05d}/{ecg_id:05d}_lr'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    for ext in ['.mat', '.hea']:  # chapman uses .mat signal files
        urllib.request.urlretrieve(PTBXL + f'records100/{folder:05d}/{ecg_id:05d}_lr{ext}',
                                    path + ext)
    return wfdb.rdrecord(path)

def chapman(rel):
    path = 'chapman/' + rel
    os.makedirs(os.path.dirname(path), exist_ok=True)
    for ext in ['.mat', '.hea']:  # chapman uses .mat signal files
        urllib.request.urlretrieve(CHAP + f'WFDBRecords/{rel}{ext}', path + ext)
    return wfdb.rdrecord(path)

def plot10(rec, title, lead=None):
    if lead is None:
        names = [n.lower() for n in rec.sig_name]
        lead = names.index('ii') if 'ii' in names else 0
    sig = rec.p_signal[:, lead]
    fs = rec.fs
    n = min(10 * fs, len(sig))
    t = np.arange(n) / fs
    plt.figure(figsize=(12, 2.5))
    plt.plot(t, sig[:n], lw=0.7, color='tab:blue')
    plt.xlabel('time (s)'); plt.ylabel('mV'); plt.grid(alpha=0.3)
    plt.title(f'{title}  ({fs} Hz, lead {rec.sig_name[lead]})')
    plt.show()
print('helpers ready')

## 1. Normal Sinus Rhythm
Regular P-QRS-T, rate 60-100 bpm. PTB-XL record 1, diagnosis NORM.

In [ ]:
plot10(ptbxl(1), 'Normal Sinus Rhythm')

## 2. Sinus Bradycardia
Normal rhythm but slow (< 60 bpm). Chapman-Shaoxing record JS00006, diagnosis SB.

In [ ]:
plot10(chapman('01/010/JS00006'), 'Sinus Bradycardia (~slow rate)')

## 3. Sinus Tachycardia
Normal rhythm but fast (> 100 bpm). PTB-XL record 103, diagnosis STACH.

In [ ]:
plot10(ptbxl(103), 'Sinus Tachycardia')

## 4. Atrial Fibrillation (AFib)
No clear P waves, irregular R-R. From afdb record 04015 - a long-term
recording with annotated AF episodes (red shading).

In [ ]:
import wfdb, os
os.makedirs('afdb', exist_ok=True)
wfdb.dl_database('afdb', dl_dir='afdb', records=['04015'])
rec = wfdb.rdrecord('afdb/04015')
ann = wfdb.rdann('afdb/04015', 'atr')
changes = [(s, n) for s, n in zip(ann.sample, ann.aux_note) if n in ('(AFIB', '(N')]
fs, sig = rec.fs, rec.p_signal[:, 0]
first_af = next(s for s, n in changes if n == '(AFIB')
start, end = max(0, first_af - 5*fs), max(0, first_af - 5*fs) + 20*fs
t = np.arange(start, end) / fs
plt.figure(figsize=(12, 2.5))
plt.plot(t, sig[start:end], lw=0.7)
for i in range(len(changes)-1):
    a, b = changes[i][0], changes[i+1][0]
    if changes[i][1] == '(AFIB' and b > start and a < end:
        plt.axvspan(max(a, start)/fs, min(b, end)/fs, color='red', alpha=0.15)
plt.xlabel('time (s)'); plt.grid(alpha=0.3)
plt.title('AFib episode start (afdb 04015, red = AF)')
plt.show()

## 5. Atrial Flutter
Sawtooth flutter waves, often regular ventricular response.
Chapman-Shaoxing record JS00005, diagnosis AF.

In [ ]:
plot10(chapman('01/010/JS00005'), 'Atrial Flutter')

## 6. Supraventricular Tachycardia (SVT)
Fast rhythm originating above the ventricles. PTB-XL record 336, SVTAC.

In [ ]:
plot10(ptbxl(336), 'Supraventricular Tachycardia')

## 7. Ventricular Tachycardia (VT)
Run of wide ventricular beats. MIT-BIH record 223 - header notes two
slow VT episodes (100-105 bpm, bidirectional).

In [ ]:
os.makedirs('mitdb', exist_ok=True)
wfdb.dl_database('mitdb', dl_dir='mitdb', records=['223'])
rec = wfdb.rdrecord('mitdb/223')
fs, sig = rec.fs, rec.p_signal[:, 0]
# find the region with the most ventricular beats (V symbols)
ann = wfdb.rdann('mitdb/223', 'atr')
v_samples = [s for s, sym in zip(ann.sample, ann.symbol) if sym == 'V']
mid = v_samples[len(v_samples)//2]
start = max(0, mid - 5*fs)
t = np.arange(start, start + 10*fs) / fs
plt.figure(figsize=(12, 2.5))
plt.plot(t, sig[start:start + 10*fs], lw=0.7)
plt.xlabel('time (s)'); plt.grid(alpha=0.3)
plt.title('Ventricular beats run (mitdb 223)')
plt.show()

## 8. Ventricular Fibrillation (VFib)
Chaotic, no organized QRS. MIT-BIH VF Database record 418 - the record
deteriorates into ventricular fibrillation.

In [ ]:
os.makedirs('vfdb', exist_ok=True)
wfdb.dl_database('vfdb', dl_dir='vfdb', records=['418'])
rec = wfdb.rdrecord('vfdb/418')
fs, sig = rec.fs, rec.p_signal[:, 0]
fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=False)
for ax, (start, label) in zip(axes, [(0, 'early in record'),
                                     (int(len(sig)*0.85), 'late in record')]):
    t = np.arange(start, start + 8*fs) / fs
    ax.plot(t, sig[start:start + 8*fs], lw=0.7)
    ax.set_title(f'vfdb 418 - {label}'); ax.grid(alpha=0.3)
plt.xlabel('time (s)'); plt.tight_layout(); plt.show()

## 9. PVC - Premature Ventricular Contraction
Early wide beat without a preceding P. PTB-XL record 170, diagnosis PVC.

In [ ]:
plot10(ptbxl(170), 'PVC')

## 10. PAC - Premature Atrial Contraction
Early beat with a (often abnormal) P wave and normal QRS.
PTB-XL record 257, diagnosis PAC.

In [ ]:
plot10(ptbxl(257), 'PAC')

## 11. 1st-Degree AV Block
PR interval longer than 200 ms but every P still conducts.
PTB-XL record 102, diagnosis 1AVB.

In [ ]:
plot10(ptbxl(102), '1st-Degree AV Block (long PR)')

## 12. 3rd-Degree (Complete) AV Block
P waves and QRS complexes are completely dissociated.
PTB-XL record 959, diagnosis 3AVB.

In [ ]:
plot10(ptbxl(959), '3rd-Degree (Complete) Heart Block')

## 13. Bundle Branch Blocks
Wide QRS from delayed conduction in one ventricle.
Left: PTB-XL 180 (CLBBB). Right: PTB-XL 172 (CRBBB).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 4.5))
for ax, (ecg_id, name) in zip(axes, [(180, 'LBBB (CLBBB)'), (172, 'RBBB (CRBBB)')]):
    r = ptbxl(ecg_id)
    names = [n.lower() for n in r.sig_name]
    lead = names.index('ii') if 'ii' in names else 0
    n = min(10*r.fs, r.sig_len)
    ax.plot(np.arange(n)/r.fs, r.p_signal[:n, lead], lw=0.7)
    ax.set_title(f'PTB-XL {ecg_id} - {name}'); ax.grid(alpha=0.3)
plt.xlabel('time (s)'); plt.tight_layout(); plt.show()

## 14. Wolff-Parkinson-White (WPW)
Short PR with a slurred QRS upstroke (delta wave) from an accessory
pathway. PTB-XL record 2145, diagnosis WPW.

In [ ]:
plot10(ptbxl(2145), 'WPW (short PR, delta wave)')

## Not available as real recordings in our datasets

These types were in the original list but none of our datasets
(PTB-XL, Chapman, MIT-BIH, afdb, vfdb) carry labeled examples:

- **AIVR** (accelerated idioventricular rhythm)
- **Torsades de Pointes** - exists in the CUDB database (not downloaded)
- **2nd-degree AV block** - rare, present in Chapman but not among the
  records checked so far
- **Parasystole** - mentioned in MIT-BIH record comments but not
  annotated as a beat class
- **Sick Sinus Syndrome** - long-term monitoring needed (afdb-style
  datasets with SSS annotations exist elsewhere)

These were drawn as simulated waveforms in the earlier notebook; for
the thesis only real-data types above are used.
